![zand_dam banner image](./seadam.png)

In [1]:
# this is the bash command to remove your virtual environment

# sudo /anaconda/bin/conda remove -n py38_selenium --all

In [2]:
# these are the commands to create the coding environment for this notebook

# sudo /anaconda/bin/conda create -y --name py38_selenium python=3.8
# conda activate /anaconda/envs/py38_selenium
# pip install ipykernel selenium webdriver-manager beautifulsoup4 lxml html5lib xlrd wget pandas holidays

In [3]:
# these commands are for installing google-chrome on ubuntu if needed

# wget -nc https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
# sudo apt update
# sudo apt install -f ./google-chrome-stable_current_amd64.deb

In [4]:
# check installed version
!chromium-browser --version

Chromium 135.0.7049.84 snap


check the version based on Release
https://chromedriver.chromium.org/downloads

In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from datetime import datetime
from holidays import Belgium
from time import sleep
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import math
import requests

### selenium v4

Since version 4 you need to use Chrome Driver Manager and it will automatically  
install the correct Chrome Driver for your environment.

In [6]:
! pip show selenium

Name: selenium
Version: 4.8.3
Summary: 
Home-page: https://www.selenium.dev
Author: 
Author-email: 
License: Apache 2.0
Location: /anaconda/envs/py38_selenium/lib/python3.8/site-packages
Requires: certifi, trio, trio-websocket, urllib3
Required-by: 


In [7]:
print ('Last testrun on: ' + datetime.now().strftime("%d %b %Y"))

Last testrun on: 23 Sep 2025


In [8]:
options = Options()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument("--window-size=1920,1200")  # adviced to increase resolution

# here is where the magic happens
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [9]:
# open the webpage where we can find all tides
driver.get("https://odnature.naturalsciences.be/marine-forecasting-centre/nl/harmonic-tides")

In [10]:
# maximize the window
driver.maximize_window()

In [11]:
# check the image (this is used for debugging code)
driver.save_screenshot('screen.png')

True

## change the start date

In [12]:
# Change the start date formaat JJJJ-MM-DD
begindatum = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.CSS_SELECTOR, "input[id='harmonic-start-date']")))

begindatum.clear() # Clear any existing value
begindatum.send_keys("2026-01-01") # CHANGE THE YEAR

## change the end date

In [13]:
# Locate the end date input field
einddatum = WebDriverWait(driver, 10).until(EC.visibility_of_element_located((By.CSS_SELECTOR, "input[id='harmonic-end-date']")))

# Click the input field to make it editable
einddatum.click()

einddatum.clear() # Clear any existing value
einddatum.send_keys("2026-12-31") # CHANGE THE YEAR

### downloading website table

In [14]:
button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.ID, "update-table")))
button.click()

In [15]:
# wait 5 seconds, for the webpage to load
sleep(5)

In [16]:
# Get the page source after all interactions
page_source = driver.page_source

In [17]:
# Parse the page source with BeautifulSoup
soup = BeautifulSoup(page_source, 'html.parser')

In [18]:
# Extract the table with the specific attribute name="tableResults"
table = soup.find('table', {'name': 'tableResults'})

In [19]:
# Convert the table to a DataFrame
df = pd.read_html(str(table))[0]

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Datum               730 non-null    object
 1   Hoog water [m TAW]  730 non-null    object
 2   Tijd [UTC]          730 non-null    object
 3   Laag water [m TAW]  730 non-null    object
 4   Tijd [UTC].1        730 non-null    object
dtypes: object(5)
memory usage: 28.7+ KB


In [21]:
df

,Datum,Hoog water [m TAW],Tijd [UTC],Laag water [m TAW],Tijd [UTC].1
0,NaN,NaN,NaN,NaN,NaN
1,2026-01-01,4.26,09:50,0.81,04:06
2,2026-01-01,4.22,22:28,0.55,16:33
3,2026-01-02,4.43,10:50,0.61,05:11
4,2026-01-02,4.36,23:24,0.47,17:33
...,...,...,...,...,...
726,2026-12-29,4.48,16:08,0.72,22:38
727,2026-12-30,4.27,04:30,0.27,11:10
728,2026-12-30,4.29,17:06,0.87,23:30
729,2026-12-31,4.14,05:28,--.--,--.--


### clean and prepare the data

In [22]:
# change the column names
df.columns = ['datum', 'hoog_water', 'hoog_tijd', 'laag_water', 'laag_tijd']

In [23]:
# remove record 0 because it is all NaN
df = df.drop(0, axis=0)

In [24]:
# first replace the values "--.--"" into NaN
x = {"--.--":np.nan}
df = df.replace(x)

# change hoog en laag water from string into float numbers
df = df.astype({'hoog_water':'float', 'laag_water':'float'})

In [25]:
# Convert 'datum' to datetime format
df['datum'] = pd.to_datetime(df['datum'])

In [26]:
# Combine the date from 'datum' with the time values from 'hoog_tijd' and 'laag_tijd'
df['hoog_tijd'] = pd.to_datetime(df['datum'].dt.strftime('%Y-%m-%d') + ' ' + df['hoog_tijd'])
df['laag_tijd'] = pd.to_datetime(df['datum'].dt.strftime('%Y-%m-%d') + ' ' + df['laag_tijd'])

In [27]:
# Localize to UTC
df['hoog_tijd'] = df['hoog_tijd'].dt.tz_localize('UTC')
df['laag_tijd'] = df['laag_tijd'].dt.tz_localize('UTC')

In [28]:
# Converteer de tijden naar de lokale tijdzone van Oostende, België ("Europe/Brussels")
df['laag_tijd'] = df['laag_tijd'].dt.tz_convert('Europe/Brussels')
df['hoog_tijd'] = df['hoog_tijd'].dt.tz_convert('Europe/Brussels')

In [29]:
# remove the column "datum"
df.drop(["datum"], axis=1, inplace=True)

In [30]:
# Get basic statistics for each column
print(df.describe())

       hoog_water  laag_water
count  705.000000  705.000000
mean     4.290383    0.519759
std      0.405535    0.381216
min      3.200000   -0.310000
25%      4.010000    0.250000
50%      4.310000    0.490000
75%      4.620000    0.790000
max      5.080000    1.500000


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 730 entries, 1 to 730
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype                          
---  ------      --------------  -----                          
 0   hoog_water  705 non-null    float64                        
 1   hoog_tijd   705 non-null    datetime64[ns, Europe/Brussels]
 2   laag_water  705 non-null    float64                        
 3   laag_tijd   705 non-null    datetime64[ns, Europe/Brussels]
dtypes: datetime64[ns, Europe/Brussels](2), float64(2)
memory usage: 22.9 KB


In [32]:
# backup/store the data into a CSV file
df.to_csv('getijden.csv', index=0)

## we now need to filter for dates that allow to build the zand-dam during low tide

In [33]:
# Filter the data to get rows where low tide is between 16:00 and 17:00
# I would like to drive in the morning arrive 9u30, then start 10u30-11u00.
# so high tide is 6 hours before low tide and it stays high for 1 hour so,
# ex: if low tide is 16:30 then we can start 11u30 ---> ~4 hours of good water flow
low_tide_filter = (df['laag_tijd'].dt.hour >= 15) & (df['laag_tijd'].dt.hour < 17)

In [34]:
# Filter voor maanden tussen mei en oktober (maanden 5 t/m 10) 
# maar negeer de maanden juli (7) en augustus (8)
month_filter = (df['laag_tijd'].dt.month >= 5) & (df['laag_tijd'].dt.month <= 10) & ~(df['laag_tijd'].dt.month.isin([7, 8]))
# month_filter = (df['laag_tijd'].dt.month >= 5) & (df['laag_tijd'].dt.month <= 10)

In [35]:
# Drop NaN values from 'laag_tijd' and get unique years
unique_years = df['laag_tijd'].dropna().dt.year.unique().tolist()

In [36]:
# Fetch Belgian holidays for the unique years
belgian_holidays = Belgium(years=unique_years)

In [37]:
belgian_holidays

{datetime.date(2026, 1, 1): 'Nieuwjaar', datetime.date(2026, 4, 5): 'Pasen', datetime.date(2026, 4, 6): 'Paasmaandag', datetime.date(2026, 5, 1): 'Dag van de Arbeid', datetime.date(2026, 5, 14): 'O. L. H. Hemelvaart', datetime.date(2026, 5, 24): 'Pinksteren', datetime.date(2026, 5, 25): 'Pinkstermaandag', datetime.date(2026, 7, 21): 'Nationale feestdag', datetime.date(2026, 8, 15): 'O. L. V. Hemelvaart', datetime.date(2026, 11, 1): 'Allerheiligen', datetime.date(2026, 11, 11): 'Wapenstilstand', datetime.date(2026, 12, 25): 'Kerstmis'}

In [38]:
# Convert the keys (dates) of the belgian_holidays dictionary to a list
feestdagen = [str(date) for date in belgian_holidays.keys()]

In [39]:
feestdag_filter = df['laag_tijd'].dt.date.astype(str).isin(feestdagen)

In [40]:
# Filter voor enkel weekenddagen: vrijdag (4), zaterdag (5), zondag (6) en maandag (0)
weekend_filter = df['laag_tijd'].dt.weekday.isin([4, 5, 6, 0])

In [41]:
# Combineer de filters met een OR-operatie
weekend_feestdag = weekend_filter | feestdag_filter

In [42]:
weekend_feestdag

1       True
2       True
3       True
4       True
5       True
       ...  
726    False
727    False
728    False
729    False
730    False
Name: laag_tijd, Length: 730, dtype: bool

In [43]:
df[low_tide_filter == True]

,hoog_water,hoog_tijd,laag_water,laag_tijd
26,3.56,2026-01-13 21:21:00+01:00,1.06,2026-01-13 15:12:00+01:00
28,3.58,2026-01-14 22:32:00+01:00,1.14,2026-01-14 16:18:00+01:00
58,3.81,2026-01-29 22:09:00+01:00,0.79,2026-01-29 16:06:00+01:00
86,3.31,2026-02-12 21:44:00+01:00,1.30,2026-02-12 15:27:00+01:00
88,3.52,2026-02-13 23:03:00+01:00,1.25,2026-02-13 16:36:00+01:00
116,3.64,2026-02-27 22:03:00+01:00,0.94,2026-02-27 15:51:00+01:00
146,3.42,2026-03-14 22:21:00+01:00,1.25,2026-03-14 16:00:00+01:00
174,3.66,2026-03-28 21:51:00+01:00,0.96,2026-03-28 15:45:00+01:00
202,3.27,2026-04-11 21:13:00+02:00,1.28,2026-04-11 15:18:00+02:00
204,3.46,2026-04-12 22:33:00+02:00,1.15,2026-04-12 16:24:00+02:00


In [44]:
df[(low_tide_filter == True) & (month_filter == True)]

,hoog_water,hoog_tijd,laag_water,laag_tijd
262,3.64,2026-05-11 21:44:00+02:00,1.03,2026-05-11 15:46:00+02:00
263,3.87,2026-05-12 22:42:00+02:00,0.87,2026-05-12 16:44:00+02:00
290,3.88,2026-05-25 21:52:00+02:00,0.90,2026-05-25 15:47:00+02:00
320,3.89,2026-06-09 20:55:00+02:00,0.91,2026-06-09 15:00:00+02:00
322,4.01,2026-06-10 21:54:00+02:00,0.82,2026-06-10 15:59:00+02:00
324,4.15,2026-06-11 22:51:00+02:00,0.74,2026-06-11 16:58:00+02:00
348,3.93,2026-06-23 21:07:00+02:00,0.92,2026-06-23 15:06:00+02:00
350,3.87,2026-06-24 22:08:00+02:00,0.96,2026-06-24 16:08:00+02:00
498,3.90,2026-09-06 21:44:00+02:00,0.97,2026-09-06 15:38:00+02:00
526,3.38,2026-09-20 21:01:00+02:00,1.35,2026-09-20 15:09:00+02:00


In [45]:
# Combine all the filters
combined_filter = low_tide_filter & month_filter & weekend_feestdag

In [46]:
# Get the days that satisfy all conditions = best days
df[combined_filter]

,hoog_water,hoog_tijd,laag_water,laag_tijd
262,3.64,2026-05-11 21:44:00+02:00,1.03,2026-05-11 15:46:00+02:00
290,3.88,2026-05-25 21:52:00+02:00,0.90,2026-05-25 15:47:00+02:00
498,3.90,2026-09-06 21:44:00+02:00,0.97,2026-09-06 15:38:00+02:00
526,3.38,2026-09-20 21:01:00+02:00,1.35,2026-09-20 15:09:00+02:00
528,3.48,2026-09-21 22:36:00+02:00,1.28,2026-09-21 16:21:00+02:00
556,3.93,2026-10-05 21:37:00+02:00,0.90,2026-10-05 15:26:00+02:00


In [47]:
max_length = max([len(row['hoog_tijd'].strftime("%A %d %B %Y")) for _, row in df[combined_filter].iterrows()])

for index, row in df[combined_filter].iterrows():
    date_str = row['hoog_tijd'].strftime("%A %d %B %Y")
    time_str = row['laag_tijd'].strftime("%Hu%M")
    print(f"{date_str:<{max_length}} (met laag water om {time_str})")

Monday 11 May 2026       (met laag water om 15u46)
Monday 25 May 2026       (met laag water om 15u47)
Sunday 06 September 2026 (met laag water om 15u38)
Sunday 20 September 2026 (met laag water om 15u09)
Monday 21 September 2026 (met laag water om 16u21)
Monday 05 October 2026   (met laag water om 15u26)
